# Gradio Demo (Member 3)

Build a web demo with the best checkpoint (default: LoRA r=4). Running it in Colab creates a public `share=True` link suitable for recording the demo video.

**Important (HANDOFF.md § 12 Gotcha 1 + 3):** the processor must use `min_pixels=256*28*28, max_pixels=768*28*28`, otherwise inference results will not match the training distribution.

In [ ]:
# Step 0: Environment
!nvidia-smi
import os, sys
# Force English locale for Python-side messages where possible.
os.environ["LANG"] = "en_US.UTF-8"
os.environ["LC_ALL"] = "en_US.UTF-8"
os.environ["LANGUAGE"] = "en_US"
if not os.path.exists('GR5293-peft-medical-vqa'):
    !git clone https://github.com/qiujunzhang03-7/GR5293-peft-medical-vqa.git
%cd GR5293-peft-medical-vqa
%pip install -q -r requirements.txt
%pip install -q gradio qwen-vl-utils

PROJECT_ROOT = '/content/GR5293-peft-medical-vqa'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [ ]:
# Step 1: Restore the best adapter from Drive (backed up after training)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BACKUP = '/content/drive/MyDrive/peft_medical_vqa/checkpoints'
!mkdir -p checkpoints
# By default, copy LoRA r=4 (Member 1 best). change to dora_rank4 or another checkpoint if cross-method analysis finds DoRA better
!cp -r {DRIVE_BACKUP}/lora_rank4 checkpoints/
!ls checkpoints/lora_rank4

In [ ]:
# Step 2: Load base model + adapter
import torch
from peft import PeftModel
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

ADAPTER_PATH = "checkpoints/lora_rank4"   # change to the best checkpoint
BASE_MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

print("Loading base model...")
base = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("Attaching LoRA adapter...")
model = PeftModel.from_pretrained(base, ADAPTER_PATH)
model.eval()

# CRITICAL: same as training-time min_pixels/max_pixels (HANDOFF.md § 12)
processor = AutoProcessor.from_pretrained(
    BASE_MODEL_ID,
    min_pixels=256*28*28,
    max_pixels=768*28*28,
)

model.print_trainable_parameters()
print("\n✅ Model + adapter loaded.")

In [ ]:
# Step 3: Prepare example images (select examples where baseline is wrong and LoRA is correct from the test set)
import json, os
from PIL import Image
from datasets import load_dataset

# Load wins from error analysis
with open('results/error_analysis/baseline_vs_lora_rank4.json') as f:
    err_data = json.load(f)

# Take the 5 most dramatic wins (prioritize closed-question bias correction)
wins = err_data.get('examples', err_data) if isinstance(err_data, dict) else err_data
if isinstance(wins, dict):
    wins = wins.get('examples', [])
    
print(f"Found {len(wins)} wins")
demo_picks = wins[:5]
for w in demo_picks:
    print(f"  Q: {w['question'][:60]}")
    print(f"  Gold: {w['reference']}, Baseline: {w.get('baseline_pred', 'N/A')}")

# Load test-set images
ds = load_dataset('flaviagiammarino/vqa-rad', split='test')
os.makedirs('demo_examples', exist_ok=True)
examples_for_gradio = []
for w in demo_picks:
    idx = w['id']
    img = ds[idx]['image']
    img_path = f"demo_examples/example_{idx}.png"
    img.save(img_path)
    examples_for_gradio.append([img_path, w['question']])

print(f"\nPrepared {len(examples_for_gradio)} demo examples")

In [ ]:
# Step 4: Build the predict function
from src.evaluation.evaluate_baseline import generate_answer
from PIL import Image

def predict(image, question):
    """Input: PIL Image + question string. Output: model answer."""
    if image is None or not question:
        return "Please upload a radiology image and enter a question."
    if not isinstance(image, Image.Image):
        image = Image.fromarray(image)
    if image.mode != 'RGB':
        image = image.convert('RGB')
    
    try:
        with torch.no_grad():
            answer = generate_answer(
                model, processor, image, question, max_new_tokens=64
            )
        return answer.strip() or "(The model did not generate an answer)"
    except Exception as e:
        return f"Error: {type(e).__name__}: {e}"

In [ ]:
# Step 5: Launch Gradio with English-only UI text (share=True creates a public link)
import gradio as gr

FORCE_ENGLISH_JS = """
() => {
  document.documentElement.setAttribute('lang', 'en');
  document.documentElement.setAttribute('translate', 'no');
  document.body.setAttribute('translate', 'no');
  const meta = document.createElement('meta');
  meta.name = 'google';
  meta.content = 'notranslate';
  document.head.appendChild(meta);
}
"""

with gr.Blocks(
    title="PEFT-Tuned Qwen2-VL on VQA-RAD",
    js=FORCE_ENGLISH_JS,
    head="""
    <meta name="google" content="notranslate">
    <meta http-equiv="content-language" content="en">
    """,
) as demo:
    gr.Markdown(
        """
        # PEFT-Tuned Qwen2-VL on VQA-RAD

        Qwen2-VL-2B fine-tuned with LoRA r=4  
        4.6M trainable parameters, 0.21% of total parameters.

        VQA-RAD test performance: Closed EM 75.7%, Open Token-F1 35.6%.  
        This demo is configured with English-only labels and text.
        """
    )
    with gr.Row():
        with gr.Column():
            image_input = gr.Image(type="pil", label="Radiology Image")
            question_input = gr.Textbox(
                label="Question",
                placeholder="e.g. is there evidence of an aortic aneurysm?",
            )
            with gr.Row():
                submit_btn = gr.Button("Generate Answer", variant="primary")
                clear_btn = gr.Button("Clear")
        with gr.Column():
            answer_output = gr.Textbox(label="Model Answer", lines=4)
    gr.Examples(
        examples=examples_for_gradio,
        inputs=[image_input, question_input],
        label="Example Questions",
    )
    submit_btn.click(
        fn=predict,
        inputs=[image_input, question_input],
        outputs=answer_output,
    )
    clear_btn.click(
        fn=lambda: (None, "", ""),
        inputs=None,
        outputs=[image_input, question_input, answer_output],
    )

demo.launch(share=True, debug=False)
